# CNN

1. Input Layer
2. Convolutional Layer
3. Pooling Layer
4. Fully Connected Layer
5. Output Layer

## Input Layer

## Convolutional Layer

입력 데이터에서 특성을 추출하는 역할을 수행

● 입력 이미지가 들어왔을 때 이미지에 대한 특성을 감지하기 위해 kernel이나 필터를 사용

● 커널/필터는 이미지의 모든 영역을 훑으면서 특성을 추출하게 되는데, 이렇게 추출된 결과물이 특성맵  (feature map)

stride : 몇 칸씩 이동하는지

3x3x4 : 3x3 짜리가 4개(4장) 있다는 뜻

### 입력 데이터: W1×H1×D1 (W1: 가로, ×H1: 세로, ×D1: 채널 또는 깊이)

### 하이퍼파라미터

• 필터 개수: K

• 필터 크기: F

• 스트라이드: S

• 패딩: P

### 출력 데이터

• W2 = (W1-F+2P)/S+1

• H2 = (H1-F+2P)/S+1

• D2 = K

## Pooling Layer

다운샘플링 (크기줄임)

max pooling(최댓값) / average pooling(평균값)

## Fully Connected Layer (완전연결층)

합성곱층과 풀링층을 거치면서 차원이 축소된 특성 맵은 최종적으로 fully connected layer으로 전달

이 과정에서 이미지는 3차원 벡터에서 1차원 벡터로 flatten 됨

## Output Layer(출력층)

Softmax 활성화 함수가 사용되는데, 입력받은 값을 0~1 사이의 값으로 출력

마지막 출력층의 소프트맥스 함수를 사용하여 이미지가 각 label에 속할 확률값이 출력되며, 이때 가장 높은 확률 값을 갖는 레이블이 최종 값으로 선정

# 실습 - MNIST FASHION

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

## 데이터 불러오기

In [2]:
train_dataset = torchvision.datasets.FashionMNIST(
    "data", download=True, transform=transforms.Compose([transforms.ToTensor()])
    )
test_dataset = torchvision.datasets.FashionMNIST(
    "data", download=True, train=False, transform=transforms.Compose([transforms.ToTensor()])
    )

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 170kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.21MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 24.3MB/s]


In [3]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100)

### labeling

In [4]:
labels_map = { 0:'T-shirt', 1:'Trouser', 2:'Pullover', 3:'Dress', 4:'Coat', 5:'Sandal', 6:'Shirt', 7:'Sneaker', 8:'Bag', 9:'Ankle Boot'}

## 모델 생성

CNN과 DNN을 비교해보자.

### CNN이 아닌 DNN

In [5]:
class FashionDNN(nn.Module):
  def __init__(self):
    super(FashionDNN, self).__init__()
    self.fc1 = nn.Linear(in_features=784, out_features=256)   # image size = 28*28 = 784
    self.drop = nn.Dropout(0.25)
    self.fc2 = nn.Linear(in_features=256, out_features=128)
    self.fc3 = nn.Linear(in_features=128, out_features=10)

  def forward(self, input_data):
    out = input_data.view(-1, 784)
    out = F.relu(self.fc1(out))
    out = self.drop(out)
    out = F.relu(self.fc2(out))
    out = self.fc3(out)
    return out

In [8]:
learning_rate = 0.001
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FashionDNN()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(model)

FashionDNN(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (drop): Dropout(p=0.25, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)


#### 학습

In [10]:
from torch.nn.modules import loss
num_epochs = 5
count = 0
loss_list = []
iteration_list = []
accuracy_list = []
predictions_list = []
labels_list = []

for epoch in range(num_epochs):

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    train = Variable(images.view(100, 1, 28, 28))
    labels = Variable(labels)
    outputs = model(train)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    count += 1
    if not (count % 50):
      total = 0
      correct = 0

      for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        labels_list.append(labels)
        test = Variable(images.view(100, 1, 28, 28))
        outputs = model(test)
        predictions = torch.max(outputs, 1)[1].to(device)
        predictions_list.append(predictions)
        correct += (predictions == labels).sum()
        total += len(labels)

      accuracy = correct * 100 / total
      loss_list.append(loss.data)
      iteration_list.append(count)
      accuracy_list.append(accuracy)

    if not (count % 500):
      print("Iteration: {}, Loss: {}, Accuracy: {}%".format(count, loss.data, accuracy))


Iteration: 500, Loss: 0.4460855722427368, Accuracy: 86.45999908447266%
Iteration: 1000, Loss: 0.3784325420856476, Accuracy: 86.1199951171875%
Iteration: 1500, Loss: 0.28739941120147705, Accuracy: 86.40999603271484%
Iteration: 2000, Loss: 0.30673080682754517, Accuracy: 86.94999694824219%
Iteration: 2500, Loss: 0.19069240987300873, Accuracy: 87.23999786376953%
Iteration: 3000, Loss: 0.2863384485244751, Accuracy: 87.63999938964844%


### CNN

In [11]:
class FashionCNN(nn.Module):

  def __init__(self):
    super(FashionCNN, self).__init__()

    self.layer1 = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.layer2 = nn.Sequential(
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.fc1 = nn.Linear(in_features=64*6*6, out_features=600)  # 12x12 가 반으로 줄어서 6x6
    self.drop = nn.Dropout2d(0.25)
    self.fc2 = nn.Linear(in_features=600, out_features=120)
    self.fc3 = nn.Linear(in_features=120, out_features=10)    # 10개의 숫자 (클래스 개수) : 0~9

  def forward(self, x):
    out = self.layer1(x)
    out = self.layer2(out)
    out = out.view(out.size(0), -1)
    out = self.fc1(out)
    out = self.drop(out)
    out = self.fc2(out)
    out = self.fc3(out)
    return out

In [12]:
learning_rate = 0.001
model = FashionCNN()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(model)

FashionCNN(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=2304, out_features=600, bias=True)
  (drop): Dropout2d(p=0.25, inplace=False)
  (fc2): Linear(in_features=600, out_features=120, bias=True)
  (fc3): Linear(in_features=120, out_features=10, bias=True)
)


#### 학습

In [13]:
num_epochs = 5
count = 0
loss_list = []
iteration_list = []
accuracy_list = []
predictions_list = []
labels_list = []

for epoch in range(num_epochs):

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    train = Variable(images.view(100, 1, 28, 28))
    labels = Variable(labels)
    outputs = model(train)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    count += 1

    if not (count % 50):
      total = 0
      correct = 0

      for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        labels_list.append(labels)
        test = Variable(images.view(100, 1, 28, 28))
        outputs = model(test)
        predictions = torch.max(outputs, 1)[1].to(device)
        predictions_list.append(predictions)
        correct += (predictions == labels).sum()
        total += len(labels)

      accuracy = correct * 100 / total
      loss_list.append(loss.data)
      iteration_list.append(count)
      accuracy_list.append(accuracy)

    if not (count % 500):
      print("Iteration: {}, Loss: {}, Accuracy: {}%".format(count, loss.data, accuracy))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  return F.dropout2d(input, self.p, self.training, self.inplace)


Iteration: 500, Loss: 0.46524107456207275, Accuracy: 88.43999481201172%
Iteration: 1000, Loss: 0.3700140714645386, Accuracy: 87.5%
Iteration: 1500, Loss: 0.3268275558948517, Accuracy: 88.68000030517578%
Iteration: 2000, Loss: 0.21499285101890564, Accuracy: 89.08999633789062%
Iteration: 2500, Loss: 0.1846485733985901, Accuracy: 90.0%
Iteration: 3000, Loss: 0.18641971051692963, Accuracy: 90.54000091552734%


# 실습 - 새로운 모델 만들어보기

- Convolution Layer 하나 추가해보기
- 다양한 hyperparameter 사용해서 성능 높이기 optimizer, loss, conv 안의 숫자 바꿔보기
- Convolutional Layer를 최대치까지 추가해보기

In [30]:
class newFashionCNN(nn.Module):

  def __init__(self):
    super(newFashionCNN, self).__init__()

    self.layer1 = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.layer2 = nn.Sequential(
        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.layer3 = nn.Sequential(
        nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.layer4 = nn.Sequential(
        nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.fc1 = nn.Linear(in_features=256*1*1, out_features=600)  # Corrected input features based on output of layer4 (256 * 1 * 1)
    self.drop = nn.Dropout2d(0.25)
    self.fc2 = nn.Linear(in_features=600, out_features=120)
    self.fc3 = nn.Linear(in_features=120, out_features=10)    # 10개의 숫자 (클래스 개수) : 0~9

  def forward(self, x):
    out = self.layer1(x)
    out = self.layer2(out)
    out = self.layer3(out)
    out = self.layer4(out) # Added layer4 to the forward pass
    out = out.view(out.size(0), -1)
    out = self.fc1(out)
    out = self.drop(out)
    out = self.fc2(out)
    out = self.fc3(out)
    return out

In [33]:
learning_rate = 0.001 # Changed learning rate for Adam
model = newFashionCNN() # Instantiating the new model
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) # Changed optimizer to Adam
print(model)

newFashionCNN(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, mo

## 학습

In [34]:
num_epochs = 15 # Increased number of epochs
count = 0
loss_list = []
iteration_list = []
accuracy_list = []
predictions_list = []
labels_list = []

for epoch in range(num_epochs):

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    train = Variable(images.view(100, 1, 28, 28))
    labels = Variable(labels)
    outputs = model(train)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    count += 1

    if not (count % 50):
      total = 0
      correct = 0

      for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        labels_list.append(labels)
        test = Variable(images.view(100, 1, 28, 28))
        outputs = model(test)
        predictions = torch.max(outputs, 1)[1].to(device)
        predictions_list.append(predictions)
        correct += (predictions == labels).sum()
        total += len(labels)

      accuracy = correct * 100 / total
      loss_list.append(loss.data)
      iteration_list.append(count)
      accuracy_list.append(accuracy)

    if not (count % 500):
      print("Iteration: {}, Loss: {}, Accuracy: {}%".format(count, loss.data, accuracy))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  return F.dropout2d(input, self.p, self.training, self.inplace)


Iteration: 500, Loss: 0.43550461530685425, Accuracy: 88.0999984741211%
Iteration: 1000, Loss: 0.32090359926223755, Accuracy: 89.32999420166016%
Iteration: 1500, Loss: 0.18400241434574127, Accuracy: 88.98999786376953%
Iteration: 2000, Loss: 0.18366952240467072, Accuracy: 89.82999420166016%
Iteration: 2500, Loss: 0.142449289560318, Accuracy: 90.29999542236328%
Iteration: 3000, Loss: 0.19527605175971985, Accuracy: 89.61000061035156%
Iteration: 3500, Loss: 0.25760918855667114, Accuracy: 90.98999786376953%
Iteration: 4000, Loss: 0.17984649538993835, Accuracy: 90.80999755859375%
Iteration: 4500, Loss: 0.05426546931266785, Accuracy: 90.69999694824219%
Iteration: 5000, Loss: 0.16588203608989716, Accuracy: 90.54999542236328%
Iteration: 5500, Loss: 0.07091119885444641, Accuracy: 91.29000091552734%
Iteration: 6000, Loss: 0.08528409898281097, Accuracy: 90.29000091552734%
Iteration: 6500, Loss: 0.13653741776943207, Accuracy: 90.12999725341797%
Iteration: 7000, Loss: 0.1084863469004631, Accuracy: 91